# Gold - State of Data Brasil (Tech Challenge)

Camada **agregada em star schema** para consumo do BI. Lê da Silver (`slv_*`) e grava em `gold/`:

| Prefixo | Tipo | Papel no BI |
|---|---|---|
| `gld_dim_*` | Dimensões | cada dimensão é **um filtro** do BI (`cd_` = código, `ds_` = descrição) |
| `gld_ft_*` | Fatos | uma tabela por tema de análise: `GROUP BY` pelos códigos das dimensões + indicadores já calculados |
| `gld_meta_constraints` | Metadados | PKs e FKs do modelo (validadas no final) |

**Perguntas -> fatos**

| Pergunta do desafio | Fato |
|---|---|
| Como está estruturado o mercado de Dados? | `gld_ft_estrutura_mercado` (profissionais) e `gld_ft_estrutura_empresas` (setor, porte, maturidade) |
| Quais perfis são mais valorizados? / diferenças por região e senioridade | `gld_ft_remuneracao` |
| Diferenças por modelo de trabalho | `gld_ft_modelo_trabalho_retencao` |
| Cenário de diversidade de gênero (e raça/PcD) | `gld_ft_diversidade` |
| Quais tecnologias têm maior adoção? | `gld_ft_adocao_opcao` |
| Adoção de IA e seu impacto | `gld_ft_ia_generativa` (+ tipos de uso e barreiras em `gld_ft_adocao_opcao`) |
| Oportunidades e desafios para as empresas (retenção, modelo de trabalho) | `gld_ft_modelo_trabalho_retencao` |

**Drill-through:** os códigos das dimensões são os mesmos da Silver (`cd_senioridade`, `cd_setor`...), então o detalhe de qualquer número da Gold se encontra filtrando `slv_evt_respondente` pelos mesmos códigos. Exceções: `cd_regiao`, `cd_cargo` (junta cargo e cargo de gestão) e `cd_uso_ia`, que são derivados e documentados abaixo.

In [ ]:
%idle_timeout 30
%worker_type G.1X
%number_of_workers 2

In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BUCKET = "tech-challenge-datahackers"
BASE   = f"s3://{BUCKET}"
SILVER = f"{BASE}/silver"
GOLD   = f"{BASE}/gold"

rd = lambda t: spark.read.parquet(f"{SILVER}/{t}/")
evt       = rd("slv_evt_respondente")
resposta  = rd("slv_evt_resposta_multipla")
mtd_opcao = rd("slv_mtd_opcao")
mtd_perg  = rd("slv_mtd_pergunta")
print("silver: respondentes =", evt.count(), "| respostas =", resposta.count())

NAO_INFORMADO = -1       # codigo usado quando o respondente nao viu/nao respondeu a pergunta
EXTERIOR      = -2       # regiao: mora fora do Brasil

CONSTRAINTS = []         # (tabela, tipo, colunas, tabela_ref, colunas_ref)
def registrar_pk(tab, cols):             CONSTRAINTS.append((tab, "PK", cols, None, None))
def registrar_fk(tab, cols, ref, rcols): CONSTRAINTS.append((tab, "FK", cols, ref, rcols))

def gravar(df, tab):
    df.coalesce(1).write.mode("overwrite").parquet(f"{GOLD}/{tab}/")

## 1. Dimensões

Toda dimensão recebe o membro **-1 = "Não informado / não se aplica"** (a Silver permite FK nula; na Gold a FK nunca é nula, assim o filtro do BI não perde linhas).

In [ ]:
def dimensao(silver_tab, gold_tab, cd, ds, extras=()):
    """Copia uma MTD da Silver mantendo os codigos e acrescenta o membro -1."""
    d = rd(silver_tab)
    cols = [cd, ds, *[c for c in extras if c in d.columns]]
    d = d.select(*cols)
    nulo = {c: (NAO_INFORMADO if c == cd else ("Não informado" if c == ds else None)) for c in cols}
    d = d.unionByName(spark.createDataFrame([tuple(nulo[c] for c in cols)], d.schema))
    gravar(d.orderBy(cd), gold_tab)
    registrar_pk(gold_tab, [cd])

dimensao("slv_mtd_senioridade",  "gld_dim_senioridade",  "cd_senioridade",  "ds_senioridade",  ["nr_ordem"])
dimensao("slv_mtd_atuacao_dados","gld_dim_atuacao",      "cd_atuacao_dados","ds_atuacao_dados")
dimensao("slv_mtd_setor",        "gld_dim_setor",        "cd_setor",        "ds_setor")
dimensao("slv_mtd_porte_empresa","gld_dim_porte_empresa","cd_porte_empresa","ds_porte_empresa",["nr_ordem"])
dimensao("slv_mtd_modelo_trabalho","gld_dim_modelo_trabalho","cd_modelo_trabalho","ds_modelo_trabalho")
dimensao("slv_mtd_genero",       "gld_dim_genero",       "cd_genero",       "ds_genero")
dimensao("slv_mtd_cor_raca_etnia","gld_dim_cor_raca_etnia","cd_cor_raca_etnia","ds_cor_raca_etnia")

# --- pesquisa (ano) ---
p = rd("slv_mtd_pesquisa").select("cd_pesquisa", "nr_ano",
        F.concat(F.lit("State of Data "), F.col("nr_ano")).alias("ds_pesquisa"))
gravar(p, "gld_dim_pesquisa"); registrar_pk("gld_dim_pesquisa", ["cd_pesquisa"])

# --- regiao (derivada da UF de residencia) ---
reg = rd("slv_mtd_regiao").select("cd_regiao", "ds_regiao")
reg = reg.unionByName(spark.createDataFrame([(NAO_INFORMADO, "Não informado"), (EXTERIOR, "Exterior")], reg.schema))
gravar(reg.orderBy("cd_regiao"), "gld_dim_regiao"); registrar_pk("gld_dim_regiao", ["cd_regiao"])

# --- cargo: cargos individuais (codigo da Silver) + cargos de gestao (codigo + 100) ---
c1 = rd("slv_mtd_cargo").select(F.col("cd_cargo"), F.col("ds_cargo"), F.lit("Contribuidor individual").alias("ds_tipo_cargo"))
c2 = rd("slv_mtd_cargo_gestor").select((F.col("cd_cargo_gestor") + 100).alias("cd_cargo"),
        F.concat(F.lit("Gestão: "), F.col("ds_cargo_gestor")).alias("ds_cargo"), F.lit("Gestão").alias("ds_tipo_cargo"))
c3 = spark.createDataFrame([(NAO_INFORMADO, "Não informado", "Não informado")], c1.schema)
gravar(c1.unionByName(c2).unionByName(c3).orderBy("cd_cargo"), "gld_dim_cargo"); registrar_pk("gld_dim_cargo", ["cd_cargo"])

# --- uso de IA generativa no trabalho (categoria derivada, em ordem de investimento) ---
USO_IA = [(NAO_INFORMADO, "Não informado", None), (1, "Não usa IA generativa", 1), (2, "Usa soluções gratuitas", 2),
          (3, "Usa Copilot", 3), (4, "Usa e paga do próprio bolso", 4), (5, "Empresa paga a solução", 5)]
gravar(spark.createDataFrame(USO_IA, "cd_uso_ia int, ds_uso_ia string, nr_ordem int"), "gld_dim_uso_ia")
registrar_pk("gld_dim_uso_ia", ["cd_uso_ia"])

# --- opcao (tecnologia / resposta de multipla escolha) com tema para filtrar no BI ---
TEMAS = {
    "s4_linguagem_de_programacao_dia_a_dia": "Linguagens de programação",
    "s4_banco_de_dados_dia_a_dia": "Bancos de dados",
    "s4_cloud_dia_a_dia": "Cloud",
    "s4_ferramenta_de_bi_dia_a_dia": "Ferramentas de BI",
    "s4_fontes_de_dados_dia_a_dia": "Fontes de dados",
    "s4_fonte_de_dado_mais_usada": "Fontes de dados",
    "s6_ferramentas_etl_de": "Ferramentas de ETL",
    "s7_ferramentas_etl_da": "Ferramentas de ETL",
    "s7_ferramentas_autonomia_area_de_negocios": "Autonomia da área de negócios",
    "s8_tecnologias_ds": "Tecnologias de ciência de dados",
    "s8_tecnicas_e_metodos_ds": "Técnicas de ciência de dados",
    "s4_usa_chatgpt_ou_copilot_no_trabalho": "IA generativa: uso individual",
    "s4_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "IA generativa: uso na empresa",
    "s3_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "IA generativa: uso na empresa",
    "s3_motivos_para_nao_usar_ai_generativa_e_llm": "IA generativa: barreiras",
    "s2_criterios_para_escolha_de_emprego": "Critérios de escolha de emprego",
    "s2_motivo_insatisfacao": "Motivos de insatisfação",
    "s1_aspectos_prejudicados": "Diversidade: impactos na carreira",
    "s1_experiencia_profissional_prejudicada": "Diversidade: impactos na carreira",
    "s3_desafios_como_gestor": "Gestão de times de dados",
    "s3_responsabilidades_como_gestor": "Gestão de times de dados",
    "s3_cargos_no_time_de_dados_da_empresa": "Gestão de times de dados",
}
tema = F.create_map(*[x for kv in TEMAS.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])
dim_opcao = (mtd_opcao.join(mtd_perg, "cd_pergunta")
    .select("cd_opcao", "cd_pergunta", "nm_pergunta", "ds_pergunta", "nr_secao", "ds_opcao",
            F.coalesce(F.element_at(tema, F.col("nm_pergunta")), F.lit("Rotina de trabalho")).alias("ds_tema")))
dim_opcao = dim_opcao.unionByName(spark.createDataFrame(
    [(NAO_INFORMADO, NAO_INFORMADO, "nao_informado", "Não informado", None, "Não informado", "Não informado")], dim_opcao.schema))
gravar(dim_opcao.orderBy("cd_opcao"), "gld_dim_opcao"); registrar_pk("gld_dim_opcao", ["cd_opcao"])
print("dimensoes gravadas")

## 2. Base de respondentes com indicadores

Uma visão intermediária (não é gravada) com os códigos das dimensões já tratados (`-1` no lugar de nulo) e as **colunas de indicador**, calculadas uma vez por respondente:
`ind_<nome>` (1 = atende ao critério) e `bas_<nome>` (1 = respondeu a pergunta, ou seja, entra no denominador).
O percentual do fato sempre usa como base **quem respondeu**, não o total de respondentes.

In [ ]:
def cds(tabela, cd, ds, predicado):
    """Codigos de uma MTD cuja descricao atende ao predicado (evita fixar codigos no codigo)."""
    return [r[0] for r in rd(tabela).select(cd, ds).collect() if predicado(r[1])]

pos_grad   = cds("slv_mtd_nivel_ensino", "cd_nivel_ensino", "ds_nivel_ensino", lambda d: d in ("Pós-graduação", "Mestrado", "Doutorado ou Phd"))
ate_2_anos = cds("slv_mtd_tempo_exp_dados", "cd_tempo_exp_dados", "ds_tempo_exp_dados", lambda d: d.startswith(("Não tenho", "Menos de 1", "de 1 a 2")))
negros     = cds("slv_mtd_cor_raca_etnia", "cd_cor_raca_etnia", "ds_cor_raca_etnia", lambda d: d in ("Preta", "Parda"))
pcd_sim    = cds("slv_mtd_pcd", "cd_pcd", "ds_pcd", lambda d: d == "Sim")
masculino  = cds("slv_mtd_genero", "cd_genero", "ds_genero", lambda d: d == "Masculino")
ia_alta    = cds("slv_mtd_prioridade_ia_generativa", "cd_prioridade_ia_generativa", "ds_prioridade_ia_generativa", lambda d: d.startswith("Sim"))
buscando   = cds("slv_mtd_plano_mudanca_6m", "cd_plano_mudanca_6m", "ds_plano_mudanca_6m", lambda d: d.startswith("Estou em busca"))
afetado    = cds("slv_mtd_layoff", "cd_layoff", "ds_layoff", lambda d: "eu fui afetado" in d)
rto_sai    = cds("slv_mtd_atitude_retorno_presencial", "cd_atitude_retorno_presencial", "ds_atitude_retorno_presencial", lambda d: d.startswith("Vou procurar"))

# uso de IA: 1 linha por respondente a partir da pergunta de multipla escolha
uso = (resposta.join(mtd_opcao, "cd_opcao").join(mtd_perg.where(F.col("nm_pergunta") == "s4_usa_chatgpt_ou_copilot_no_trabalho"), "cd_pergunta")
       .groupBy("sk_respondente")
       .agg(*[F.max(F.when((F.col("fl_selecionado") == 1) & F.col("ds_opcao").contains(txt), 1).otherwise(0)).alias(k)
              for k, txt in [("empresa", "A empresa que trabalho paga"), ("paga", "Uso e pago"), ("copilot", "Copilot"),
                             ("gratuita", "gratuitas"), ("nao_usa", "Não uso")]]))
cd_uso_ia = (F.when(F.col("empresa") == 1, 5).when(F.col("paga") == 1, 4).when(F.col("copilot") == 1, 3)
              .when(F.col("gratuita") == 1, 2).when(F.col("nao_usa") == 1, 1).otherwise(NAO_INFORMADO))

nvl = lambda c: F.coalesce(F.col(c), F.lit(NAO_INFORMADO))
sal = F.col("vl_salario_medio")

base = (evt
    .join(rd("slv_mtd_uf").select("cd_uf", "cd_regiao"), evt.cd_uf_residencia == F.col("cd_uf"), "left")
    .join(uso, "sk_respondente", "left")
    .select(
        "sk_respondente", "cd_pesquisa",
        F.when(F.col("cd_regiao").isNotNull(), F.col("cd_regiao"))
         .when(F.col("fl_vive_no_brasil") == False, F.lit(EXTERIOR)).otherwise(F.lit(NAO_INFORMADO)).alias("cd_regiao"),
        nvl("cd_atuacao_dados").alias("cd_atuacao_dados"),
        F.coalesce(F.col("cd_cargo"), F.col("cd_cargo_gestor") + 100, F.lit(NAO_INFORMADO)).alias("cd_cargo"),
        nvl("cd_senioridade").alias("cd_senioridade"),
        nvl("cd_setor").alias("cd_setor"),
        nvl("cd_porte_empresa").alias("cd_porte_empresa"),
        nvl("cd_modelo_trabalho_atual").alias("cd_modelo_trabalho_atual"),
        nvl("cd_modelo_trabalho_ideal").alias("cd_modelo_trabalho_ideal"),
        nvl("cd_genero").alias("cd_genero"),
        nvl("cd_cor_raca_etnia").alias("cd_cor_raca_etnia"),
        F.coalesce(cd_uso_ia, F.lit(NAO_INFORMADO)).alias("cd_uso_ia"),
        sal.alias("vl_salario"),
        # --- indicadores (ind_) e bases (bas_) ---
        F.col("fl_atua_como_gestor").cast("int").alias("ind_gestor"),             F.col("fl_atua_como_gestor").isNotNull().cast("int").alias("bas_gestor"),
        F.col("cd_nivel_ensino").isin(pos_grad).cast("int").alias("ind_pos_graduado"), F.col("cd_nivel_ensino").isNotNull().cast("int").alias("bas_pos_graduado"),
        F.col("cd_tempo_exp_dados").isin(ate_2_anos).cast("int").alias("ind_ate_2_anos_exp"), F.col("cd_tempo_exp_dados").isNotNull().cast("int").alias("bas_ate_2_anos_exp"),
        F.col("cd_cor_raca_etnia").isin(negros).cast("int").alias("ind_negros"),   F.col("cd_cor_raca_etnia").isNotNull().cast("int").alias("bas_negros"),
        F.col("cd_pcd").isin(pcd_sim).cast("int").alias("ind_pcd"),                F.col("cd_pcd").isNotNull().cast("int").alias("bas_pcd"),
        F.col("fl_satisfeito_atualmente").cast("int").alias("ind_satisfeitos"),    F.col("fl_satisfeito_atualmente").isNotNull().cast("int").alias("bas_satisfeitos"),
        F.col("cd_plano_mudanca_6m").isin(buscando).cast("int").alias("ind_buscando_mudar"), F.col("cd_plano_mudanca_6m").isNotNull().cast("int").alias("bas_buscando_mudar"),
        F.col("cd_layoff").isin(afetado).cast("int").alias("ind_afetados_layoff"), F.col("cd_layoff").isNotNull().cast("int").alias("bas_afetados_layoff"),
        F.col("cd_atitude_retorno_presencial").isin(rto_sai).cast("int").alias("ind_sairia_retorno_presencial"),
        F.col("cd_atitude_retorno_presencial").isNotNull().cast("int").alias("bas_sairia_retorno_presencial"),
        (F.col("cd_modelo_trabalho_atual") == F.col("cd_modelo_trabalho_ideal")).cast("int").alias("ind_modelo_alinhado"),
        (F.col("cd_modelo_trabalho_atual").isNotNull() & F.col("cd_modelo_trabalho_ideal").isNotNull()).cast("int").alias("bas_modelo_alinhado"),
        F.col("fl_possui_data_lake").cast("int").alias("ind_data_lake"),           F.col("fl_possui_data_lake").isNotNull().cast("int").alias("bas_data_lake"),
        F.col("fl_possui_data_warehouse").cast("int").alias("ind_data_warehouse"), F.col("fl_possui_data_warehouse").isNotNull().cast("int").alias("bas_data_warehouse"),
        F.col("cd_prioridade_ia_generativa").isin(ia_alta).cast("int").alias("ind_ia_prioridade_alta"),
        F.col("cd_prioridade_ia_generativa").isNotNull().cast("int").alias("bas_ia_prioridade_alta"),
        (F.coalesce(cd_uso_ia, F.lit(NAO_INFORMADO)) >= 2).cast("int").alias("ind_usa_ia"),
        (F.coalesce(cd_uso_ia, F.lit(NAO_INFORMADO)) >= 1).cast("int").alias("bas_usa_ia"),
        # faixas de remuneracao (pelo ponto medio da faixa: 4k, 8k e 16k coincidem com limites de faixa da pesquisa)
        (sal < 4000).cast("int").alias("ind_sal_ate_4mil"),
        ((sal >= 4000) & (sal < 8000)).cast("int").alias("ind_sal_4_a_8mil"),
        ((sal >= 8000) & (sal < 16000)).cast("int").alias("ind_sal_8_a_16mil"),
        (sal >= 16000).cast("int").alias("ind_sal_acima_16mil"),
        sal.isNotNull().cast("int").alias("bas_salario"),
    ).cache())
print("base:", base.count(), "respondentes")

## 3. Funções de agregação

- `ind(nomes)`: para cada indicador gera `qt_<nome>`, `qt_base_<nome>` e `pct_<nome>` (percentual sobre quem respondeu).
- `salario()`: quantidade, soma, média e mediana. A "média" é a média dos pontos médios das faixas salariais (a pesquisa só informa faixas).
- `pct_participacao_pesquisa`: participação da linha no total do ano. Soma 100% por ano, então **pode ser somado no BI sob qualquer filtro**.
- `qt_` e `qt_base_` são aditivos (o BI pode somá-los); `pct_` é válido no grão do fato.

In [ ]:
def ind(nomes):
    cols = []
    for n in nomes:
        cols += [F.sum(f"ind_{n}").alias(f"qt_{n}"), F.sum(f"bas_{n}").alias(f"qt_base_{n}")]
    return cols

def pct(df, nomes):
    for n in nomes:
        df = df.withColumn(f"pct_{n}", F.when(F.col(f"qt_base_{n}") > 0, F.round(100 * F.col(f"qt_{n}") / F.col(f"qt_base_{n}"), 1)))
    return df

def salario():
    return [F.count("vl_salario").alias("qt_com_salario"), F.round(F.sum("vl_salario"), 2).alias("vl_salario_soma"),
            F.round(F.avg("vl_salario"), 2).alias("vl_salario_medio"),
            F.expr("percentile_approx(vl_salario, 0.5)").alias("vl_salario_mediano")]

def participacao(df):
    return df.withColumn("pct_participacao_pesquisa",
                         F.round(100 * F.col("qt_respondentes") / F.sum("qt_respondentes").over(Window.partitionBy("cd_pesquisa")), 2))

FAIXAS = ["sal_ate_4mil", "sal_4_a_8mil", "sal_8_a_16mil", "sal_acima_16mil"]

def fato(nome, df, dims):
    gravar(df, nome)
    registrar_pk(nome, dims)
    for d in dims:
        dim = {"cd_pesquisa": "gld_dim_pesquisa", "cd_regiao": "gld_dim_regiao", "cd_atuacao_dados": "gld_dim_atuacao",
               "cd_cargo": "gld_dim_cargo", "cd_senioridade": "gld_dim_senioridade", "cd_setor": "gld_dim_setor",
               "cd_porte_empresa": "gld_dim_porte_empresa", "cd_modelo_trabalho_atual": "gld_dim_modelo_trabalho",
               "cd_modelo_trabalho_ideal": "gld_dim_modelo_trabalho", "cd_genero": "gld_dim_genero",
               "cd_cor_raca_etnia": "gld_dim_cor_raca_etnia", "cd_uso_ia": "gld_dim_uso_ia", "cd_opcao": "gld_dim_opcao"}[d]
        registrar_fk(nome, [d], dim, ["cd_modelo_trabalho" if dim == "gld_dim_modelo_trabalho" else d])
    print(f"{nome}: {df.count()} linhas")

## 4. Fatos

In [ ]:
# ---- F1: estrutura do mercado (profissionais) ----------------------------------------------
dims = ["cd_pesquisa", "cd_regiao", "cd_atuacao_dados", "cd_senioridade"]
nomes = ["gestor", "pos_graduado", "ate_2_anos_exp"]
f1 = base.groupBy(*dims).agg(F.count("*").alias("qt_respondentes"), *salario(), *ind(nomes))
f1 = pct(participacao(f1), nomes)
fato("gld_ft_estrutura_mercado", f1, dims)

# ---- F1b: estrutura do mercado (empresas: setor, porte e maturidade em dados / IA) ----
dims = ["cd_pesquisa", "cd_setor", "cd_porte_empresa"]
nomes = ["data_lake", "data_warehouse", "ia_prioridade_alta", "satisfeitos", "buscando_mudar"]
f1b = base.groupBy(*dims).agg(F.count("*").alias("qt_respondentes"), *salario(), *ind(nomes))
f1b = pct(participacao(f1b), nomes)
fato("gld_ft_estrutura_empresas", f1b, dims)

# ---- F2: remuneracao (perfis mais valorizados e diferencas por regiao / senioridade) ----
dims = ["cd_pesquisa", "cd_regiao", "cd_cargo", "cd_senioridade"]
f2 = base.groupBy(*dims).agg(F.count("*").alias("qt_respondentes"), *salario(),
                             *[F.sum(f"ind_{n}").alias(f"qt_{n}") for n in FAIXAS])
f2 = participacao(f2)
for n in FAIXAS:     # base das faixas = quem informou salario
    f2 = f2.withColumn(f"pct_{n}", F.when(F.col("qt_com_salario") > 0, F.round(100 * F.col(f"qt_{n}") / F.col("qt_com_salario"), 1)))
fato("gld_ft_remuneracao", f2, dims)

In [ ]:
# ---- F3: diversidade (genero, raca/cor, PcD) + diferenca salarial vs. homens ----
dims = ["cd_pesquisa", "cd_genero", "cd_cor_raca_etnia", "cd_atuacao_dados", "cd_senioridade"]
nomes = ["negros", "pcd"]
f3 = base.groupBy(*dims).agg(F.count("*").alias("qt_respondentes"), *salario(), *ind(nomes))
f3 = pct(participacao(f3), nomes)
# participacao no grupo (mesma pesquisa, atuacao e senioridade): somavel entre as linhas do mesmo grupo
f3 = f3.withColumn("pct_participacao_no_grupo", F.round(100 * F.col("qt_respondentes") /
        F.sum("qt_respondentes").over(Window.partitionBy("cd_pesquisa", "cd_atuacao_dados", "cd_senioridade")), 1))
# referencia: salario medio dos homens no mesmo grupo
ref = (base.where(F.col("cd_genero").isin(masculino))
       .groupBy("cd_pesquisa", "cd_atuacao_dados", "cd_senioridade")
       .agg(F.round(F.avg("vl_salario"), 2).alias("vl_salario_medio_ref_masculino")))
f3 = (f3.join(ref, ["cd_pesquisa", "cd_atuacao_dados", "cd_senioridade"], "left")
        .withColumn("pct_dif_salario_vs_masculino",
                    F.when(F.col("vl_salario_medio_ref_masculino") > 0,
                           F.round(100 * (F.col("vl_salario_medio") / F.col("vl_salario_medio_ref_masculino") - 1), 1))))
fato("gld_ft_diversidade", f3, dims)

# ---- F4: adocao de tecnologias e respostas de multipla escolha ----
dims = ["cd_pesquisa", "cd_opcao", "cd_atuacao_dados", "cd_senioridade"]
f4 = (resposta.join(base.select("sk_respondente", "cd_pesquisa", "cd_atuacao_dados", "cd_senioridade", "vl_salario"), "sk_respondente")
      .groupBy(*dims)
      .agg(F.count("*").alias("qt_base"),                                   # respondentes que viram a pergunta
           F.sum("fl_selecionado").alias("qt_selecionado"),
           F.sum(F.when(F.col("fl_selecionado") == 1, F.col("vl_salario"))).alias("_soma"),
           F.sum(F.when((F.col("fl_selecionado") == 1) & F.col("vl_salario").isNotNull(), 1).otherwise(0)).alias("qt_selecionado_com_salario"))
      .withColumn("pct_adocao", F.round(100 * F.col("qt_selecionado") / F.col("qt_base"), 1))
      .withColumn("vl_salario_soma_selecionado", F.round("_soma", 2))
      .withColumn("vl_salario_medio_selecionado",
                  F.when(F.col("qt_selecionado_com_salario") > 0, F.round(F.col("_soma") / F.col("qt_selecionado_com_salario"), 2)))
      .drop("_soma"))
fato("gld_ft_adocao_opcao", f4, dims)

In [ ]:
# ---- F5: IA generativa (adocao e impacto em remuneracao, satisfacao e mobilidade) ----
dims = ["cd_pesquisa", "cd_uso_ia", "cd_atuacao_dados", "cd_senioridade"]
nomes = ["satisfeitos", "buscando_mudar", "ia_prioridade_alta"]
f5 = base.groupBy(*dims).agg(F.count("*").alias("qt_respondentes"), *salario(), *ind(nomes))
f5 = pct(participacao(f5), nomes)
fato("gld_ft_ia_generativa", f5, dims)

# ---- F6: modelo de trabalho, satisfacao e retencao ----
dims = ["cd_pesquisa", "cd_regiao", "cd_senioridade", "cd_modelo_trabalho_atual", "cd_modelo_trabalho_ideal"]
nomes = ["satisfeitos", "buscando_mudar", "afetados_layoff", "sairia_retorno_presencial", "modelo_alinhado"]
f6 = base.groupBy(*dims).agg(F.count("*").alias("qt_respondentes"), *salario(), *ind(nomes))
f6 = pct(participacao(f6), nomes)
fato("gld_ft_modelo_trabalho_retencao", f6, dims)

## 5. Validação

1. **PK**: sem nulos e sem duplicidade (a PK dos fatos é a combinação dos códigos das dimensões).
2. **FK**: todo código do fato existe na dimensão (nenhuma FK nula na Gold).
3. **Reconciliação com a Silver**: o total agregado da Gold tem que bater com o detalhe granular (por ano), garantindo que o drill-through não aponta para números diferentes.

In [ ]:
meta = [(t, tp, ",".join(c), rt, ",".join(rc) if rc else None) for t, tp, c, rt, rc in CONSTRAINTS]
gravar(spark.createDataFrame(meta, "tabela string, tipo string, colunas string, tabela_ref string, colunas_ref string"), "gld_meta_constraints")

def validar():
    falhas, linhas = [], []
    for tab, tipo, cols, rtab, rcols in CONSTRAINTS:
        df = spark.read.parquet(f"{GOLD}/{tab}/")
        if tipo == "PK":
            n = df.count(); distintos = df.select(*cols).distinct().count()
            nulos = df.where(" OR ".join(f"{c} IS NULL" for c in cols)).count()
            ok = (n == distintos and nulos == 0)
            linhas.append((tab, f"PK({len(cols)} col)", f"{n} linhas, {n - distintos} duplicadas, {nulos} nulas", ok))
        else:
            ref = spark.read.parquet(f"{GOLD}/{rtab}/").select(F.col(rcols[0]).alias(cols[0])).distinct()
            orfas = df.select(*cols).join(ref, cols, "left_anti").count()
            ok = (orfas == 0)
            linhas.append((tab, f"FK({cols[0]}) -> {rtab}", f"{orfas} orfas", ok))
        if not ok: falhas.append(linhas[-1])
    for t, c, d, ok in linhas:
        print(f"{'OK' if ok else 'FALHA':6} {t:34} {c:58} {d}")
    assert not falhas, f"{len(falhas)} constraint(s) violadas"
    print(f"\n{len(linhas)} constraints verificadas, todas OK")

validar()

In [ ]:
# reconciliacao Gold x Silver, por ano
silver_n = {r[0]: r[1] for r in evt.groupBy("cd_pesquisa").count().collect()}
for ft in ["gld_ft_estrutura_mercado", "gld_ft_estrutura_empresas", "gld_ft_remuneracao", "gld_ft_diversidade", "gld_ft_ia_generativa", "gld_ft_modelo_trabalho_retencao"]:
    g = {r[0]: r[1] for r in spark.read.parquet(f"{GOLD}/{ft}/").groupBy("cd_pesquisa").agg(F.sum("qt_respondentes")).collect()}
    assert g == silver_n, f"{ft}: gold {g} != silver {silver_n}"
    print(f"OK  {ft}: {g}")

sel_silver = {r[0]: r[1] for r in resposta.join(evt.select("sk_respondente", "cd_pesquisa"), "sk_respondente")
              .where("fl_selecionado = 1").groupBy("cd_pesquisa").count().collect()}
sel_gold = {r[0]: r[1] for r in spark.read.parquet(f"{GOLD}/gld_ft_adocao_opcao/").groupBy("cd_pesquisa").agg(F.sum("qt_selecionado")).collect()}
assert sel_gold == sel_silver, f"gld_ft_adocao_opcao: gold {sel_gold} != silver {sel_silver}"
print(f"OK  gld_ft_adocao_opcao (selecionados): {sel_gold}")

## 6. Conferência: consultas que o BI vai fazer

Exemplos de leitura do star schema (fato + dimensões), um por pergunta do desafio.

In [ ]:
g = lambda t: spark.read.parquet(f"{GOLD}/{t}/")
d_pes, d_sen, d_cargo, d_gen, d_uso, d_op = (g("gld_dim_pesquisa"), g("gld_dim_senioridade"), g("gld_dim_cargo"),
                                             g("gld_dim_genero"), g("gld_dim_uso_ia"), g("gld_dim_opcao"))

print("Q2 - perfis mais valorizados (2025, cargos com >= 30 respondentes com salario)")
(g("gld_ft_remuneracao").join(d_pes, "cd_pesquisa").join(d_cargo, "cd_cargo").where("nr_ano = 2025")
  .groupBy("ds_cargo").agg(F.sum("qt_com_salario").alias("n"), F.round(F.sum("vl_salario_soma") / F.sum("qt_com_salario"), 0).alias("salario_medio"))
  .where("n >= 30").orderBy(F.desc("salario_medio")).show(8, truncate=False))

print("Q3 - diversidade de genero por senioridade (2025)")
(g("gld_ft_diversidade").join(d_pes, "cd_pesquisa").join(d_sen, "cd_senioridade").join(d_gen, "cd_genero").where("nr_ano = 2025")
  .groupBy("ds_senioridade").agg(F.round(100 * F.sum(F.when(F.col("ds_genero") == "Feminino", F.col("qt_respondentes"))) / F.sum("qt_respondentes"), 1).alias("pct_feminino"),
                                 F.sum("qt_respondentes").alias("n")).orderBy("ds_senioridade").show(truncate=False))

print("Q4 - tecnologias mais adotadas: linguagens, 2025 (pct_adocao no grao atuacao x senioridade, somado pela base)")
(g("gld_ft_adocao_opcao").join(d_pes, "cd_pesquisa").join(d_op, "cd_opcao")
  .where((F.col("nr_ano") == 2025) & (F.col("ds_tema") == "Linguagens de programação"))
  .groupBy("ds_opcao").agg(F.round(100 * F.sum("qt_selecionado") / F.sum("qt_base"), 1).alias("pct_adocao")).orderBy(F.desc("pct_adocao")).show(6, truncate=False))

print("Q5 - impacto da IA: salario e satisfacao por nivel de uso (2025)")
(g("gld_ft_ia_generativa").join(d_pes, "cd_pesquisa").join(d_uso, "cd_uso_ia").where("nr_ano = 2025")
  .groupBy("ds_uso_ia", "nr_ordem").agg(F.sum("qt_respondentes").alias("n"),
        F.round(F.sum("vl_salario_soma") / F.sum("qt_com_salario"), 0).alias("salario_medio"),
        F.round(100 * F.sum("qt_satisfeitos") / F.sum("qt_base_satisfeitos"), 1).alias("pct_satisfeitos")).orderBy("nr_ordem").show(truncate=False))